# 1단계: 기준 모델과 주 비교

SCD(4년 내 급사) 예측 임상 ML 모델을 **같은 교차검증 fold**에서 학습해 비교합니다. 규칙은 `kfold/settings.py`, 전체 설명은 `kfold/README.md`에 있습니다.

| 순서 | 노트북 | 내용 |
|---|---|---|
| 0 | `ml_0_dataset` | 학습 데이터셋 결정 (705명 vs 원래 445명 출신 → 445명). class weight를 Optuna로 함께 튜닝 (결과 `kfold_logloss`) |
| 1 | `ml_1_main` | class weight 고정으로 학습하고 모델 비교 (결과 `kfold_logloss_balanced`). 6번에서 최종 모델 1개 선택 |
| 2 | `ml_2_test_all_models` | 참고: 0·1의 445명 모델 전체 test 성능 (모델 선택에 쓰지 않음) |
| 3 | `ml_3_final_test` | 최종 모델 1개 재학습과 test 1회 평가 (최종 모델을 고른 뒤 실행) |

**이 노트북에서 하는 일**
`ml_0_dataset`에서 학습 데이터셋을 정한 뒤 실행합니다.

1. 기준 모델(Pro-BNP 단독 로지스틱) 학습
2. 모델 4종 × 변수 묶음 4종 = 16개 학습 (모델별로 한 셀씩)
3. AUROC 요약표와 자세한 비교표 (6번에서 최종 모델 1개 선택)

## 0. 준비

In [ ]:
import sys
from pathlib import Path

# 저장소 루트를 찾아 import 경로에 추가합니다 (노트북을 어느 폴더에서 열어도 동작).
repo = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Members/mingyu/model/ML/kfold").is_dir()
)
sys.path.insert(0, str(repo))

from Members.mingyu.model.ML.kfold import (  # noqa: E402
    Method,
    Settings,
    auroc_grid,
    compare,
    describe,
    load_cohort,
    plot_probability_hist,
    run_method,
    save_or_check_folds,
)  # noqa: E402

settings = Settings()
cohort = load_cohort(settings)
save_or_check_folds(cohort, settings)  # 처음엔 fold를 저장하고, 이후엔 같은 fold인지 확인

print(
    f"불러온 개발 데이터 {len(cohort.ids)}명 | 이 노트북에서 쓰는 환자: 원래 445명 출신 {cohort.original.sum()}명 "
    f"(SCD {cohort.y[cohort.original].sum()}) | test {len(cohort.test_ids)}명"
)
print("결과 폴더:", settings.result_dir)

## 1. 기준 모델: Pro-BNP 단독 로지스틱

`TRAIN_FILTER`에 `ml_0_dataset`에서 정한 학습 데이터셋을 적습니다. 모든 모델이 이 데이터셋으로 학습합니다.

임상에서 가장 쉽게 쓸 수 있는 위험 지표입니다. 이후 모든 비교표의 Δ는 이 모델 대비 값입니다.

In [ ]:
TRAIN_FILTER = "only445"  # ml_0_dataset 결정: "all"(705명) 또는 "only445"(원래 445명 출신)
POPULATION = (
    "all" if TRAIN_FILTER == "all" else "original"
)  # 비교표의 평가 환자 (445명이면 445명 출신 356명)

reference = run_method(cohort, Method("Logistic", "ProBNP", TRAIN_FILTER), settings)
describe(settings, cohort, [reference])

## 2. 주 비교: 모델 4종 × 변수 묶음 4종

| 변수 묶음 | 개수 | 내용 |
|---|---|---|
| `v5` | 27 | 팀 임상 변수 23 + Holter 4 |
| `v4` | 23 | 팀 기본 임상 변수 |
| `core` | 12 | 주요 임상 변수 + Holter 3 |
| `MUSIC_SCD` | 5 | MUSIC SCD 위험 점수 항목 |

- 학습 환자: `TRAIN_FILTER` (705명이면 fold당 564명, 445명 출신이면 약 285명). 445명 출신이면 튜닝과 평가도 356명 안에서 합니다.
- 튜닝: 로지스틱은 C 9개, 트리 모델은 Optuna 100 trial. 기준은 OOF log loss
- class weight: 튜닝하지 않고 fold마다 학습 환자의 SCD 비율로 고정 (모든 모델, `Settings.class_weight`)
- 결과 폴더는 `kfold_logloss_balanced`로, `ml_0_dataset`(`kfold_logloss`)과 따로 학습합니다. 이미 학습된 방법은 건너뜁니다.

In [ ]:
FEATURE_SETS_MAIN = ["v5", "v4", "core", "MUSIC_SCD"]
MODELS_MAIN = ["Logistic", "XGBoost", "Random Forest", "CatBoost"]

### 2-1. Logistic

C(규제 세기) 후보 9개를 모두 시험합니다. 몇 분 안에 끝납니다.

In [ ]:
names = [
    run_method(cohort, Method("Logistic", features, TRAIN_FILTER), settings)
    for features in FEATURE_SETS_MAIN
]
describe(settings, cohort, names)

### 2-2. XGBoost

Optuna 100 trial × 15 fold.

In [ ]:
names = [
    run_method(cohort, Method("XGBoost", features, TRAIN_FILTER), settings)
    for features in FEATURE_SETS_MAIN
]
describe(settings, cohort, names)

### 2-3. Random Forest

Optuna 100 trial × 15 fold.

In [ ]:
names = [
    run_method(cohort, Method("Random Forest", features, TRAIN_FILTER), settings)
    for features in FEATURE_SETS_MAIN
]
describe(settings, cohort, names)

### 2-4. CatBoost

Optuna 100 trial × 15 fold. 네 모델 중 가장 오래 걸립니다.

In [ ]:
names = [
    run_method(cohort, Method("CatBoost", features, TRAIN_FILTER), settings)
    for features in FEATURE_SETS_MAIN
]
describe(settings, cohort, names)

## 3. 결과 요약: AUROC (OOF, 3반복 평균)

행 = 모델, 열 = 변수 묶음. 평가 환자는 `TRAIN_FILTER`의 환자입니다 (705명, 또는 445명 출신 356명). 아직 학습하지 않은 칸은 비어 있습니다.

In [ ]:
auroc_grid(settings, cohort, MODELS_MAIN, FEATURE_SETS_MAIN, TRAIN_FILTER)

## 4. 자세한 비교표

- AUROC, AP: 3반복 평균 ± SD
- Δ (95% CI): 기준 모델 대비, 같은 환자를 묶어 resampling한 paired bootstrap (1000회)
- 민감도/특이도: threshold 0.5 고정
- AUROC(445출신): test와 같은 집단인 356명에서의 AUROC

445명 출신으로 학습하면 평가 환자도 445명 출신 356명입니다. `comparison_main_<평가 환자>.csv`로도 저장됩니다.

In [ ]:
main = [
    Method(model, features, TRAIN_FILTER).name
    for model in MODELS_MAIN
    for features in FEATURE_SETS_MAIN
]
compare(
    settings, cohort, [reference, *main], reference=reference, population=POPULATION, save_as="main"
)

## 5. 예측 확률 분포

방법마다 학습이 끝나면 `<결과 폴더>/<이름>/probability_hist.png`가 자동으로 저장됩니다. 여기서는 17개를 같은 축으로 모아 봅니다.

- x축: 교차검증 예측 확률 (환자마다 3반복 평균), y축: 환자 수
- 색: 실제 결과 (파랑 = SCD 없음, 주황 = SCD), 점선: threshold 0.5

In [ ]:
plot_probability_hist(settings, cohort, [reference, *main], save_as="main");

## 6. 최종 모델 선택 (1개)

**선택 기준 (결과를 보기 전에 정함)**
1. OOF AUROC가 높은 순 (445명으로 정해지면 445명 출신 356명 기준)
2. 상위 모델과의 차이가 작으면(ΔAUROC 95% CI가 0을 포함) 더 단순한 모델(변수가 적은 모델, 로지스틱)을 고름

고른 모델 1개로 `ml_3_final_test`를 진행합니다. 이후 결과를 보고 모델을 바꾸지 않습니다.

- 최종 모델:
- 근거:
